# Linear SVM (soft-margin)

Encontra o hiperplano que maximiza a margem entre classes, permitindo
violacoes controladas pelo parametro `C`.

Resolvemos o problema primal direto com sub-gradient descent na
**hinge loss**, sem precisar de um solver QP.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
class LinearSVM:
    """
    SVM linear soft-margin treinado por sub-gradient descent na hinge loss.

    Objetivo:
        min 0.5 * ||w||^2 + C * sum_i max(0, 1 - y_i (w.x_i + b))

    com y em {-1, +1}. C controla o trade-off margem/erro.
    """

    def __init__(self, C=1.0, lr=0.01, n_iters=1000, random_state=None):
        if C <= 0:
            raise ValueError("C must be positive.")
        if lr <= 0:
            raise ValueError("lr must be positive.")

        self.C = C
        self.lr = lr
        self.n_iters = n_iters
        self.random_state = random_state

        self.w = None
        self.b = 0.0
        self.classes_ = None
        self.loss_history_ = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y).ravel()
        if X.ndim != 2:
            raise ValueError("X must be 2D.")

        self.classes_ = np.unique(y)
        if len(self.classes_) != 2:
            raise ValueError("LinearSVM supports binary classification only.")

        # mapeia para {-1, +1}
        y_pm = np.where(y == self.classes_[1], 1, -1).astype(float)

        n_samples, n_features = X.shape
        rng = np.random.default_rng(self.random_state)
        self.w = rng.normal(0, 0.01, size=n_features)
        self.b = 0.0
        self.loss_history_ = []

        for _ in range(self.n_iters):
            margins = y_pm * (X @ self.w + self.b)
            mask = margins < 1  # amostras que violam a margem

            grad_w = self.w - self.C * (X[mask].T @ y_pm[mask])
            grad_b = -self.C * y_pm[mask].sum()

            self.w -= self.lr * grad_w / n_samples
            self.b -= self.lr * grad_b / n_samples

            loss = 0.5 * np.dot(self.w, self.w) + self.C * np.maximum(0, 1 - margins).sum()
            self.loss_history_.append(float(loss))

        return self

    def decision_function(self, X):
        return np.asarray(X, dtype=float) @ self.w + self.b

    def predict(self, X):
        if self.w is None:
            raise ValueError("Call fit() before predict().")
        scores = self.decision_function(X)
        return np.where(scores >= 0, self.classes_[1], self.classes_[0])

    def score(self, X, y):
        return np.mean(self.predict(X) == np.asarray(y))

In [ ]:
data = datasets.load_breast_cancer()
X = StandardScaler().fit_transform(data.data)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

svm = LinearSVM(C=1.0, lr=0.01, n_iters=1000, random_state=42)
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
plt.figure()
plt.plot(svm.loss_history_)
plt.title("Hinge loss over iterations")
plt.xlabel("iteration")
plt.ylabel("loss")
plt.show()